# 01 · The observed ocean

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/blob/145a73e500cb06070f399a040206802cc279174e/content/01_observed_ocean.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/01_observed_ocean.ipynb` at `145a73e500cb06070f399a040206802cc279174e`; original SHA-256 `a118e00614ad1ed8bcbd1d8d80d4dadf19ce9a5d68d37054ad564f51db3fff53`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 01 · The observed ocean

[Run this notebook in your browser ↗](https://jltobias.github.io/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/lab/index.html?path=01_observed_ocean.ipynb)

**Evidence: published observations with modeled classifications.** Explore 139,189 source cells from the 2017–2021 Sentinel-1 study. Learn to distinguish normalized activity, an unmatched fraction, and image coverage.

**Sources:** [Paolo et al. (2024)](https://doi.org/10.1038/s41586-023-06825-8), [analysis data](https://doi.org/10.6084/m9.figshare.24309475) (CC BY 4.0); Natural Earth coastline (public domain). These are historical study products, not live AIS or satellite detections.

Run the setup cell first. Browser Python downloads numerical packages on first use. Core exercises use bundled files; live access is optional. Export your notebook to preserve work.

In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "pandas", "matplotlib", "scipy", "plotly"])
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import oceanlab as lab
from IPython.display import display
print("Runtime:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)

## Read the evidence ledger
The compressed table preserves selected source values and missingness. Coordinates are cell centers: original index / 5 + 0.1°. Fractional values reflect normalized and model-weighted activity, not fractions of physical ships.

In [ ]:
observed = lab.activity(full=True)
display(observed.head())
display(pd.DataFrame({"missing_cells": observed.isna().sum()}))
print(f"{len(observed):,} source cells; coverage: 2017–2021")

## Map the activity
The default world map groups 0.2° source cells into 1° bins and sums normalized activity. This proxy is **not fishing hours, catch, or unique vessels**. Longitude–latitude bins have unequal physical areas. Empty areas are not zero activity. Change `REGION` to `(-20, 0, 0, 25)` for West Africa or `(-12, 48, 32, 66)` for Northern Europe.

In [ ]:
REGION = (-180, -60, 180, 80)  # west, south, east, north in degrees
fig = lab.map2d("tot_fishing", bbox=REGION)
plt.show()

In [ ]:
lab.show(lab.interactive_map())

## The denominator changes the story
Compute the unmatched modeled fishing share yourself. Do not use the continent table's precomputed `perc_dark_*` or `dark_fishing_frac` columns. A minimum-activity screen prevents tiny denominators from dominating this descriptive comparison; it does not establish statistical significance.

In [ ]:
MIN_ACTIVITY = 1.0
region = observed[observed.lon.between(-12,32) & observed.lat.between(48,66)].copy()
region["unmatched_share"] = region.dark_fishing / region.tot_fishing.where(region.tot_fishing > 0)
enough = region[region.tot_fishing >= MIN_ACTIVITY]
display(enough[["lon", "lat", "tot_fishing", "dark_fishing", "unmatched_share", "overpasses_2017_2021"]].sort_values("tot_fishing", ascending=False).head(12))
print("Cells retained:", len(enough), "of", len(region), "in the teaching window")
fig, ax = plt.subplots()
p = ax.scatter(enough.overpasses_2017_2021, enough.unmatched_share,
               c=np.log10(enough.tot_fishing), cmap="viridis", s=12, alpha=.6)
ax.set(xlabel="Source imagery overpasses, 2017–2021", ylabel="Unmatched modeled fishing share", ylim=(0,1),
       title="OBSERVED STUDY PRODUCT | share, coverage and denominator")
fig.colorbar(p, ax=ax, label="log10 normalized fishing activity")
plt.show()

## Compare components across study regions
The continent figures use the deposit's aggregate estimates. They do not enumerate unique vessels, indicate violations, or describe all fleets. Compare the matched and unmatched **components**, not the source's ambiguously named fraction fields.

In [ ]:
continents = pd.read_csv(lab.DATA / "vessels_bycontinent_v20230803.csv")
cols = ["continent", "AIS fishing", "dark fishing"]
display(continents[cols].round(2))
ax = continents.set_index("continent")[["AIS fishing", "dark fishing"]].plot.barh(stacked=True, color=[lab.TEAL, lab.AMBER])
ax.set(xlabel="Study's aggregate normalized fishing activity", title="OBSERVED STUDY PRODUCT | public tracking misses part of the picture")
ax.legend(["AIS-matched component", "Unmatched component"])
plt.tight_layout(); plt.show()

## Interpret before exporting
1. Which claims depend on the sampling design?
2. How would equal-area bins change a density comparison?
3. Would an unmatched fraction identify intentional concealment?
4. What additional data would connect this activity to a fish stock?

**Takeaway:** observations make activity visible; denominators and coverage determine which comparisons are defensible.